<a href="https://colab.research.google.com/github/baddasolti619/Ir_System/blob/main/Boolean_IR_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Basic Information Retrieval System
### DCS-400 — Artificial Intelligence and Machine Learning
**Author:** Rahul

This notebook implements a small Boolean Information Retrieval (IR) system end to end:
1. Add documents to a collection
2. Preprocess text (tokenize, lowercase, remove stop words)
3. Build a dictionary (vocabulary)
4. Build an inverted index (term → posting list)
5. Evaluate Boolean queries (`AND`, `OR`, `NOT`)

This notebook is self-contained — the 8 sample documents are embedded as strings below,
so it runs top-to-bottom in Google Colab with no file uploads required.

In [ ]:
import re
import string
from collections import defaultdict

## 1. Document Collection

Eight short documents on related but distinguishable AI/ML and information-retrieval
topics, so that Boolean queries have meaningful overlap and distinguishing terms.

In [ ]:
documents_raw = {
    1: ("doc1.txt", "Artificial intelligence is the simulation of human intelligence by machines. "
                     "Machine learning is a subset of artificial intelligence that enables systems "
                     "to learn from data."),
    2: ("doc2.txt", "Deep learning uses neural networks with many layers to model complex patterns "
                     "in data. Deep learning has driven major progress in computer vision and "
                     "natural language processing."),
    3: ("doc3.txt", "Natural language processing allows computers to understand and generate human "
                     "language. Applications include machine translation, sentiment analysis, and "
                     "chatbots."),
    4: ("doc4.txt", "Information retrieval is the process of obtaining relevant information from a "
                     "large collection of documents. Search engines rely heavily on information "
                     "retrieval techniques."),
    5: ("doc5.txt", "An inverted index maps terms to the documents that contain them. It is a core "
                     "data structure used in information retrieval systems for fast search."),
    6: ("doc6.txt", "Boolean retrieval models represent documents and queries using boolean logic. "
                     "Queries combine terms with AND, OR, and NOT operators to retrieve matching "
                     "documents."),
    7: ("doc7.txt", "Computer vision enables machines to interpret and understand visual information "
                     "from the world. Convolutional neural networks are widely used for image "
                     "classification tasks."),
    8: ("doc8.txt", "Data preprocessing is an essential step before applying machine learning "
                     "algorithms. It includes tokenization, stop word removal, and stemming of "
                     "text data."),
}

for doc_id, (name, text) in documents_raw.items():
    print(f"[{doc_id}] {name}: {text[:70]}...")

[1] doc1.txt: Artificial intelligence is the simulation of human intelligence by mac...
[2] doc2.txt: Deep learning uses neural networks with many layers to model complex p...
[3] doc3.txt: Natural language processing allows computers to understand and generat...
[4] doc4.txt: Information retrieval is the process of obtaining relevant information...
[5] doc5.txt: An inverted index maps terms to the documents that contain them. It is...
[6] doc6.txt: Boolean retrieval models represent documents and queries using boolean...
[7] doc7.txt: Computer vision enables machines to interpret and understand visual in...
[8] doc8.txt: Data preprocessing is an essential step before applying machine learni...


## 2. The Information Retrieval System

The `InformationRetrievalSystem` class handles document ingestion, preprocessing,
dictionary/index construction, and Boolean query evaluation.

In [ ]:
STOP_WORDS = {
    "a", "an", "the", "is", "are", "was", "were", "be", "been", "being",
    "of", "in", "on", "at", "to", "for", "and", "or", "not", "with",
    "by", "from", "as", "that", "this", "it", "its", "into", "than",
    "such", "these", "those", "has", "have", "had", "do", "does", "did",
    "so", "if", "then", "which", "their", "them", "they", "we", "you",
}


class InformationRetrievalSystem:
    """A small Boolean Information Retrieval engine over a document set."""

    def __init__(self, use_stopwords=True):
        self.use_stopwords = use_stopwords
        self.documents = {}          # doc_id -> raw text
        self.doc_names = {}          # doc_id -> original filename
        self.dictionary = set()      # vocabulary of unique terms
        self.inverted_index = defaultdict(set)  # term -> set of doc_ids
        self._next_id = 1

    # ---- 1. Adding documents ----
    def add_document(self, text, name=None):
        doc_id = self._next_id
        self._next_id += 1
        self.documents[doc_id] = text
        self.doc_names[doc_id] = name or f"doc{doc_id}"
        self._index_document(doc_id, text)
        return doc_id

    # ---- 2. Preprocessing ----
    def tokenize(self, text):
        text = text.lower()
        text = text.translate(str.maketrans("", "", string.punctuation))
        tokens = re.findall(r"[a-z0-9]+", text)
        if self.use_stopwords:
            tokens = [t for t in tokens if t not in STOP_WORDS]
        return tokens

    # ---- 3 & 4. Dictionary + inverted index ----
    def _index_document(self, doc_id, text):
        tokens = self.tokenize(text)
        for token in set(tokens):
            self.dictionary.add(token)
            self.inverted_index[token].add(doc_id)

    def all_doc_ids(self):
        return set(self.documents.keys())

    def print_dictionary(self):
        print(f"Dictionary size: {len(self.dictionary)} unique terms")
        print(", ".join(sorted(self.dictionary)))

    def print_inverted_index(self):
        print(f"{'TERM':<15}{'POSTINGS (doc IDs)'}")
        print("-" * 40)
        for term in sorted(self.inverted_index):
            postings = sorted(self.inverted_index[term])
            print(f"{term:<15}{postings}")

    # ---- 5. Boolean retrieval ----
    def _term_postings(self, term):
        return set(self.inverted_index.get(term.lower(), set()))

    def boolean_and(self, term1, term2):
        return self._term_postings(term1) & self._term_postings(term2)

    def boolean_or(self, term1, term2):
        return self._term_postings(term1) | self._term_postings(term2)

    def boolean_not(self, term):
        return self.all_doc_ids() - self._term_postings(term)

    def search(self, query):
        """Evaluate a Boolean query string: terms plus AND / OR / NOT.
        NOT (unary) is resolved first, then AND/OR are folded left to right."""
        tokens = query.strip().split()
        if not tokens:
            return set()

        tokens = [t if t.upper() not in ("AND", "OR", "NOT") else t.upper()
                  for t in tokens]

        resolved = []
        i = 0
        while i < len(tokens):
            tok = tokens[i]
            if tok == "NOT":
                term = tokens[i + 1]
                resolved.append(("SET", self.boolean_not(term)))
                i += 2
            elif tok in ("AND", "OR"):
                resolved.append(tok)
                i += 1
            else:
                resolved.append(("SET", self._term_postings(tok)))
                i += 1

        result = resolved[0][1]
        i = 1
        while i < len(resolved):
            op = resolved[i]
            next_set = resolved[i + 1][1]
            if op == "AND":
                result = result & next_set
            elif op == "OR":
                result = result | next_set
            i += 2

        return result

    def describe_results(self, doc_ids):
        return [f"{doc_id} ({self.doc_names[doc_id]})" for doc_id in sorted(doc_ids)]

## 3. Load the Collection

Add every document to the system. Each `add_document` call immediately tokenizes the
text and updates the dictionary and inverted index.

In [ ]:
ir = InformationRetrievalSystem(use_stopwords=True)
for doc_id, (name, text) in documents_raw.items():
    ir.add_document(text, name=name)

print(f"Loaded {len(ir.documents)} documents into the system.")

Loaded 8 documents into the system.


## 4. Dictionary (Vocabulary)

In [ ]:
ir.print_dictionary()

Dictionary size: 92 unique terms
algorithms, allows, analysis, applications, applying, artificial, before, boolean, chatbots, classification, collection, combine, complex, computer, computers, contain, convolutional, core, data, deep, documents, driven, enables, engines, essential, fast, generate, heavily, human, image, include, includes, index, information, intelligence, interpret, inverted, language, large, layers, learn, learning, logic, machine, machines, major, many, maps, matching, model, models, natural, networks, neural, obtaining, operators, patterns, preprocessing, process, processing, progress, queries, relevant, rely, removal, represent, retrieval, retrieve, search, sentiment, simulation, stemming, step, stop, structure, subset, systems, tasks, techniques, terms, text, tokenization, translation, understand, used, uses, using, vision, visual, widely, word, world


## 5. Inverted Index

Each term maps to the sorted list of document IDs it appears in.

In [ ]:
ir.print_inverted_index()

TERM           POSTINGS (doc IDs)
----------------------------------------
algorithms     [8]
allows         [3]
analysis       [3]
applications   [3]
applying       [8]
artificial     [1]
before         [8]
boolean        [6]
chatbots       [3]
classification [7]
collection     [4]
combine        [6]
complex        [2]
computer       [2, 7]
computers      [3]
contain        [5]
convolutional  [7]
core           [5]
data           [1, 2, 5, 8]
deep           [2]
documents      [4, 5, 6]
driven         [2]
enables        [1, 7]
engines        [4]
essential      [8]
fast           [5]
generate       [3]
heavily        [4]
human          [1, 3]
image          [7]
include        [3]
includes       [8]
index          [5]
information    [4, 5, 7]
intelligence   [1]
interpret      [7]
inverted       [5]
language       [2, 3]
large          [4]
layers         [2]
learn          [1]
learning       [1, 2, 8]
logic          [6]
machine        [1, 3, 8]
machines       [1, 7]
major          [2]
man

## 6. Boolean Retrieval Demo

`AND` intersects postings, `OR` unions them, and `NOT` takes the complement against the
full document collection. Multi-operator queries are evaluated left to right after all
`NOT`s are resolved.

In [ ]:
demo_queries = [
    "machine AND learning",
    "vision OR language",
    "learning AND NOT deep",
    "information AND retrieval AND NOT boolean",
    "neural OR index",
]

for q in demo_queries:
    results = ir.search(q)
    print(f"Query: {q!r:45} -> {ir.describe_results(results)}")

Query: 'machine AND learning'                        -> ['1 (doc1.txt)', '8 (doc8.txt)']
Query: 'vision OR language'                          -> ['2 (doc2.txt)', '3 (doc3.txt)', '7 (doc7.txt)']
Query: 'learning AND NOT deep'                       -> ['1 (doc1.txt)', '8 (doc8.txt)']
Query: 'information AND retrieval AND NOT boolean'   -> ['4 (doc4.txt)', '5 (doc5.txt)']
Query: 'neural OR index'                             -> ['2 (doc2.txt)', '5 (doc5.txt)', '7 (doc7.txt)']


## 7. Try Your Own Query

Edit the `query` string below and re-run the cell to test any Boolean combination of
terms from the dictionary (see Section 4 for the full vocabulary).

In [ ]:
query = "computer AND vision"  # <-- edit me
results = ir.search(query)
print(f"Query: {query!r}")
print(f"Matching documents: {ir.describe_results(results)}")

Query: 'computer AND vision'
Matching documents: ['2 (doc2.txt)', '7 (doc7.txt)']


## 8. Sample Query Walkthrough

Consider `learning AND NOT deep`:
- `NOT deep` → the postings for `deep` are `{2}`, so `NOT deep` = all documents minus `{2}`.
- Intersect with `learning`'s postings `{1, 2, 8}`.
- Result: `{1, 8}` — documents 1 and 8 mention learning without being about deep learning
  specifically, while document 2 (deep learning) is correctly excluded.

## 9. Design Notes

- **Sets for postings** give fast `AND`/`OR`/`NOT` via intersection, union, and difference,
  at the cost of not tracking term positions (so phrase queries are out of scope).
- **No external NLP dependencies** — only the Python standard library, so this notebook
  runs anywhere, including a fresh Colab runtime.
- **Left-to-right operator folding** (after resolving `NOT`) is sufficient for the flat
  Boolean queries this assignment requires.

## 10. Limitations

- No ranked retrieval (e.g. TF-IDF) — only unordered Boolean matching.
- No phrase or proximity search (postings store document IDs only, not positions).
- No parenthesized/nested Boolean expressions.
